In [ ]:
import numpy as np
import pydicom
from pathlib import Path
import dicom_image_tools as dit
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
import plotly.graph_objects as go

dp = Path()

for item in dp.iterdir():
    print(item.name)


data = dit.import_dicom_from_folder(dp)

for study in data:
    for serie in data[study].Series:
        serie.import_image()
        serie.sort_images_on_acquisition_time()

study = list(data.keys())[0]

In [ ]:
def _calculate_image_lims_exclude_cornerbox(image_array):
    mins = []
    maxs = []

    xlims = [10, 25]
    ylims = [9, 24]

    if len(image_array.shape) == 2:
        image_array = image_array.astype("float")
        image_array[xlims[0] : xlims[1], ylims[0] : ylims[1]] = np.nan

        return [np.nanmin(image_array), np.nanmax(image_array)]

    for i in range(image_array.shape[0]):
        sub_array = image_array[i, :, :]
        sub_array = sub_array.astype("float")

        sub_array[
            xlims[0] : xlims[1],
            ylims[0] : ylims[1],
        ] = np.nan

        mins.append(np.nanmin(sub_array))
        maxs.append(np.nanmax(sub_array))

    lims = [
        min(mins),
        max(maxs),
    ]
    return lims

In [ ]:
imgs = data[study].Series[0].ImageVolume

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.offline import plot as plotly_plot  # offline-plot till webbläsare

rows, cols = 2, 5

# Hämta höjd och bredd från första bilden
img_h, img_w = imgs[0].shape
aspect = img_h / img_w   # automatisk beräkning av bildens aspektkvot

fig = make_subplots(
    rows=rows,
    cols=cols,
    horizontal_spacing=0.0,
    vertical_spacing=0.0
)

for idx, arr in enumerate(imgs[: rows * cols]):
    r = idx // cols + 1
    c = idx % cols + 1

    lims = _calculate_image_lims_exclude_cornerbox(imgs[idx])

    fig.add_trace(
        go.Heatmap(
            z=arr,
            zmin=lims[0],
            zmax=lims[1],
            colorscale="Gray",
            zsmooth=False,
            showscale=False
        ),
        row=r, col=c
    )

    # Gör pixlarna kvadratiska i varje subplot
    fig.update_xaxes(
        scaleanchor=f'y{idx+1}',  # kvadratiska pixlar per subplot
        scaleratio=1,
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        row=r, col=c
    )
    fig.update_yaxes(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        row=r, col=c
    )

# 🔗 Länka zoom/pan mellan ALLA subplots
# (delar samma x- respektive y-range, så zoom i en påverkar övriga)
fig.update_xaxes(matches='x')
fig.update_yaxes(matches='y')

# Auto-layout baserad på bildens aspect ratio (minimerar whitespace, behåller kvadratiska pixlar)
fig_width = 1600
fig_height = (fig_width / cols) * rows * aspect

fig.update_layout(
    width=fig_width,
    height=fig_height,
    margin=dict(l=10, r=10, t=20, b=10),
)

# 🚀 Öppna i webbläsaren (Plotly Offline)
plotly_plot(fig, auto_open=True)     # detta skapar en temporär HTML och öppnar standardwebbläsaren

# Alternativt (om du vill explicit):
# fig.show(renderer="browser")


